# ORPO Training on Qwen2.5-0.5B (Week 3)

**Revised after a failed first attempt.** Originally started from the raw base model, deliberately, to test ORPO's "monolithic, no prior SFT needed" claim — that attempt diverged catastrophically (train loss ~3.1e11, final perplexity `nan`, generations collapsed to `"!!!!!!!"`), even after lowering the learning rate 5x and reducing batch_size to 1. Most likely cause: the odds-ratio term (`log(P/(1-P))`) blows up when the untrained base model assigns near-zero probability to full chat-formatted completions — a numerical instability present from the very first loss computation, not something a smaller learning rate can fix. Documented as a genuine negative finding (Article 9): the theory doesn't guarantee from-scratch training stability, only that no *architectural* reference model is required.

This version starts from **Week 2's SFT checkpoint** instead (same as `week3_dpo_training.ipynb`), to test whether that's really the fix.

**Requires the Week 2 SFT checkpoint attached as input data** (Add Input → the `w02-sfttraining-qwen2-5-0-5b` notebook's output).

**Before running:** Settings → Accelerator → GPU T4 x2.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# ORPOTrainer imports more of transformers than SFTTrainer/DPOTrainer do (including
# transformers.integrations.flex_attention), which triggers torch.compiler.disable and,
# on this environment, an incomplete torch._dynamo install (AttributeError: module
# 'torch' has no attribute '_utils'). Disabling dynamo up front avoids that import path.
os.environ["TORCHDYNAMO_DISABLE"] = "1"
# Processing both chosen and rejected per example pushes memory close to the T4's limit;
# a lot of the OOM is fragmentation (memory PyTorch already reserved but can't reuse for
# a differently-sized allocation), which this setting lets the allocator work around.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

## 1. Pull the repo and install dependencies

Includes the `transformers`/`torchao` version fix (needed here too — `trl.experimental.orpo` has the same dependency chain), and an `HF_TOKEN` secret so dataset/model downloads aren't rate-limited as an unauthenticated request.

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
repo_url = f"https://{github_token}@github.com/zoom-BT/llm-alignment-internship.git"

!git clone --filter=blob:none --no-checkout {repo_url}
%cd llm-alignment-internship
!git sparse-checkout init --cone
!git sparse-checkout set src
!git checkout main
!pip install -q -r requirements.txt
!pip install -q -U transformers accelerate peft trl huggingface_hub torchao

## 2. Confirm the GPU is visible

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 3. Dry run (25 steps)

`config.yaml`'s `orpo.eval_steps` is 20, so 25 steps guarantees one real evaluation fires. This time `model_path` is set explicitly to the Week 2 SFT checkpoint. **Watch the logged metrics closely** — if `Nll Loss`/`Log Odds Ratio`/`Rewards/*` are finite numbers (not `nan`), the SFT-checkpoint hypothesis is confirmed.

In [ ]:
import yaml
from src.train import run_orpo

config = yaml.safe_load(open("config.yaml"))
sft_checkpoint = "/kaggle/input/notebooks/balbinotchoutzine/w02-sfttraining-qwen2-5-0-5b/llm-alignment-internship/results/checkpoints/final"

trainer = run_orpo(config, model_path=sft_checkpoint, max_steps=25)
print("Dry run finished without OOM, and eval ran at least once.")

## 4. Full ORPO run

Same 1000-pair subset as DPO, for a like-for-like comparison between the two preference methods.

In [ ]:
trainer = run_orpo(config, model_path=sft_checkpoint)
print("ORPO training complete. Model saved to results/orpo_checkpoints/final")

## 4a. Display the training curves

In [ ]:
from IPython.display import Image, display

display(Image(filename="results/orpo/training_curve.png"))

## 4b. Dolly perplexity — an out-of-domain check, not a like-for-like number

ORPO's loss includes an explicit SFT term, so it's fair to ask whether it still generalizes to instruction-following. **Caveat, same as DPO's notebook:** ORPO trained on `ultrafeedback_binarized` (general-domain), not Dolly-15k — so this number tests out-of-domain generalization, and is **not directly comparable** to Week 1-3's Dolly-trained perplexities (13.92 / 75.41 / 14.61 / 13.795), even though it uses the same `evaluate.py` code path.

In [ ]:
from src.evaluate import run_benchmark

orpo_results = run_benchmark(
    config, model_path="results/orpo_checkpoints/final", output_filename="orpo_week3_results.json"
)
orpo_results

## 4c. Qualitative check — SFT vs. ORPO (not base model, now that ORPO starts from the SFT checkpoint)

Same memory-safe ordering as the DPO notebook (generate with the already-loaded model first, free it, only then load the second model) — `run_benchmark()` above already loaded and released a third copy, so an explicit cache clear first is cheap insurance.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.evaluate import generate_samples
from src.utils import get_device

torch.cuda.empty_cache()

prompts_raw = [
    "Write a short story where a bear goes to the beach.",
    "What are the pros and cons of remote work?",
    "Explain how a car engine works.",
]

prompts = [
    trainer.processing_class.apply_chat_template(
        [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True
    )
    for p in prompts_raw
]
with torch.no_grad():
    orpo_completions = generate_samples(
        trainer.model, trainer.processing_class, prompts, max_new_tokens=150, do_sample=False
    )

del trainer
torch.cuda.empty_cache()

sft_tok = AutoTokenizer.from_pretrained(sft_checkpoint)
sft_model = AutoModelForCausalLM.from_pretrained(sft_checkpoint, dtype=torch.bfloat16)
sft_model.to(get_device())
with torch.no_grad():
    sft_completions = generate_samples(sft_model, sft_tok, prompts, max_new_tokens=150, do_sample=False)
del sft_model
torch.cuda.empty_cache()

for p, before, after in zip(prompts_raw, sft_completions, orpo_completions):
    print("=" * 80)
    print(f"PROMPT: {p}")
    print(f"-- SFT (before ORPO) --\n{before}")
    print(f"-- ORPO (after) --\n{after}")

## 5. Next step

3 of Week 3's practical methods now done (SFT+LoRA, DPO, ORPO). Left: QLoRA (to complete the full-FT vs. LoRA vs. QLoRA comparison the contract asks for), and, time permitting, an evaluator and a small GRPO/RLOO/REINFORCE reproduction.